# Benchmark Analysis Skeleton

This notebook is a starter for comparing **Design A (microservices)** vs **Design B (monolith-per-node)** benchmark results.

## 1) Setup

In [ ]:
from __future__ import annotations

from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

## 2) Notebook Config

In [ ]:
# Update these paths once real benchmark outputs exist.
DESIGN_A_ROOT = Path("../results/loadgen/design_a_placeholder")
DESIGN_B_ROOT = Path("../results/loadgen/design_b_placeholder")
EXPORT_DIR = Path("./_artifacts")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Design A root:", DESIGN_A_ROOT)
print("Design B root:", DESIGN_B_ROOT)
print("Export dir:", EXPORT_DIR.resolve())

## 3) Load Helpers

In [ ]:
def discover_run_dirs(scenario_root: Path) -> list[Path]:
    if not scenario_root.exists():
        return []
    return sorted([p for p in scenario_root.iterdir() if p.is_dir()])


def load_rows_csv(run_dir: Path) -> pd.DataFrame:
    rows_path = run_dir / "rows.csv"
    if not rows_path.exists():
        return pd.DataFrame()
    df = pd.read_csv(rows_path)
    df["run_dir"] = str(run_dir)
    return df


def load_metadata(run_dir: Path) -> dict:
    metadata_path = run_dir / "metadata.json"
    if not metadata_path.exists():
        return {}
    return json.loads(metadata_path.read_text(encoding="utf-8"))


def load_design_rows(design_root: Path, design_label: str) -> pd.DataFrame:
    parts = []
    for run_dir in discover_run_dirs(design_root):
        part = load_rows_csv(run_dir)
        if part.empty:
            continue
        part["design_label"] = design_label
        parts.append(part)
    if not parts:
        return pd.DataFrame()
    return pd.concat(parts, ignore_index=True)

## 4) Load Raw Data

In [ ]:
df_a = load_design_rows(DESIGN_A_ROOT, "A_microservices")
df_b = load_design_rows(DESIGN_B_ROOT, "B_monolith")
df = pd.concat([df_a, df_b], ignore_index=True) if (not df_a.empty or not df_b.empty) else pd.DataFrame()

print("rows_a:", len(df_a))
print("rows_b:", len(df_b))
print("rows_total:", len(df))
df.head()

## 5) Sanity Checks

In [ ]:
if df.empty:
    print("No data loaded yet. Update DESIGN_A_ROOT / DESIGN_B_ROOT once benchmark runs are available.")
else:
    print(df[["design", "scenario_id", "method"]].drop_duplicates().sort_values(["design", "scenario_id", "method"]).to_string(index=False))

## 6) Aggregate Metrics

In [ ]:
if df.empty:
    agg = pd.DataFrame()
else:
    df2 = df.copy()
    df2["is_ok"] = (df2["grpc_code"] == "OK").astype(int)
    agg = (
        df2.groupby(["design", "scenario_id", "method"], as_index=False)
        .agg(
            total_calls=("method", "count"),
            ok_calls=("is_ok", "sum"),
            p50_latency_ms=("latency_ms", lambda s: s.quantile(0.50)),
            p95_latency_ms=("latency_ms", lambda s: s.quantile(0.95)),
            p99_latency_ms=("latency_ms", lambda s: s.quantile(0.99)),
        )
    )

agg.head(20)

## 7) Plot: p95 Latency by Method

In [ ]:
if agg.empty:
    print("No aggregated data yet.")
else:
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=agg, x="method", y="p95_latency_ms", hue="design", ax=ax)
    ax.set_title("p95 Latency by Method")
    ax.set_ylabel("Latency (ms)")
    ax.set_xlabel("Method")
    plt.xticks(rotation=20)
    plt.tight_layout()
    out = EXPORT_DIR / "p95_latency_by_method.png"
    fig.savefig(out, dpi=150)
    print("saved:", out)
    plt.show()

## 8) Plot: OK Call Count by Method

In [ ]:
if agg.empty:
    print("No aggregated data yet.")
else:
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=agg, x="method", y="ok_calls", hue="design", ax=ax)
    ax.set_title("OK Calls by Method")
    ax.set_ylabel("Calls")
    ax.set_xlabel("Method")
    plt.xticks(rotation=20)
    plt.tight_layout()
    out = EXPORT_DIR / "ok_calls_by_method.png"
    fig.savefig(out, dpi=150)
    print("saved:", out)
    plt.show()

## 9) Next Steps

- Replace placeholder roots with real scenario output directories.\n
- Add scenario-level filtering/faceting and run-to-run spread/error bars.\n
- Add throughput normalization using `measure_seconds` from metadata.